In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from model import GraphSAGE
np.random.seed(42)

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = torch.load("models/task_a_model.pt", weights_only = False)
data = checkpoint['data']
student_id_map = checkpoint['student_id_map']

In [3]:
model = GraphSAGE(256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

GraphSAGE(
  (conv1): ModuleDict(
    (enrolled_in): GATConv((-1, -1), 256, heads=1)
    (engaged_with): GATConv((-1, -1), 256, heads=1)
    (prereq_of): SAGEConv(-1, 256, aggr=mean)
    (covers): SAGEConv(-1, 256, aggr=mean)
    (concept_prereq): SAGEConv(-1, 256, aggr=mean)
    (rev_enrolled_in): GATConv((-1, -1), 256, heads=1)
    (rev_engaged_with): GATConv((-1, -1), 256, heads=1)
  )
  (conv2): ModuleDict(
    (rev_enrolled_in): SAGEConv(256, 256, aggr=mean)
    (rev_engaged_with): SAGEConv(256, 256, aggr=mean)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (student_proj): Linear(in_features=5, out_features=256, bias=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (task_a_head): Linear(in_features=1024, out_features=3, bias=True)
)

In [4]:
with torch.no_grad():
    out = model(data)
    student_embs = out[0]
print("Student embeddings shape:", student_embs.shape)

Student embeddings shape: torch.Size([2000, 256])


In [5]:
enrollments = pd.read_csv('datasets/enrollments.csv')
courses = pd.read_csv('datasets/courses.csv')
students = pd.read_csv('datasets/students.csv')

course_modality = courses.set_index('course_id')['best_modality'].to_dict()
enrollments['course_modality'] = enrollments['course_id'].map(course_modality)

grade_by_modality = enrollments.groupby(['student_id','course_modality'])['grade'].mean().unstack(fill_value=0)

overall_avg = grade_by_modality.mean(axis = 1)

for col in ['analytical', 'exploratory', 'hands_on', 'visual']:
    grade_by_modality[col] = grade_by_modality[col] - overall_avg

grade_by_modality = grade_by_modality.reindex(students['student_id'])

print(grade_by_modality[['analytical', 'exploratory', 'hands_on', 'visual']].head(10))
print(f"\nShape: {grade_by_modality.shape}")

course_modality  analytical  exploratory  hands_on    visual
student_id                                                  
S0000             -0.036572    -0.144754  0.198580 -0.017254
S0001              0.027166    -0.084373 -0.030087  0.087294
S0002              0.184773    -0.231591 -0.001591  0.048409
S0003              0.498214    -1.392897  0.404246  0.490437
S0004              0.022083    -0.005774 -0.014107 -0.002202
S0005              0.017333    -0.149667 -0.078000  0.210333
S0006             -0.025911    -0.220196  0.233804  0.012304
S0007              0.012428    -0.123726 -0.079976  0.191274
S0008              0.009417     0.061417 -0.090583  0.019750
S0009              0.771292    -1.975708  0.567625  0.636792

Shape: (2000, 4)


In [6]:
modality_map = {'visual': 0, 'hands_on': 1, 'analytical': 2, 'exploratory': 3}
modality_labels = torch.tensor(
    students['true_modality'].map(modality_map).values,
    dtype= torch.long
)

In [7]:
train_mask = torch.tensor(students['split'] == 'train', dtype = torch.bool)
val_mask = torch.tensor(students['split'] == 'val', dtype = torch.bool)
test_mask = torch.tensor(students['split'] == 'test', dtype = torch.bool)

print(f"Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")

Train: 1400, Val: 300, Test: 300


In [8]:
print("Labels shape:", modality_labels.shape)
print("Distribution:")
print(students['true_modality'].value_counts())
print(f"\nTrain: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")

Labels shape: torch.Size([2000])
Distribution:
true_modality
visual         566
hands_on       516
analytical     513
exploratory    405
Name: count, dtype: int64

Train: 1400, Val: 300, Test: 300


In [9]:
features = torch.tensor(
    grade_by_modality[['analytical', 'exploratory', 'hands_on', 'visual']].values,
    dtype = torch.float
)

In [10]:
combined_features = torch.cat([student_embs, features], dim = 1)
print("Combined shape: ", combined_features.shape)

Combined shape:  torch.Size([2000, 260])


In [11]:
modality_model =  nn.Sequential(
    nn.Linear(260, 64), 
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 4)
)

optimizer = torch.optim.Adam(modality_model.parameters(), lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1000):
    modality_model.train()
    optimizer.zero_grad()

    preds = modality_model(combined_features[train_mask])
    loss = loss_fn(preds, modality_labels[train_mask])

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 1.3982
Epoch 100 | Loss: 1.2323
Epoch 200 | Loss: 0.9211
Epoch 300 | Loss: 0.6109
Epoch 400 | Loss: 0.4376
Epoch 500 | Loss: 0.3345
Epoch 600 | Loss: 0.2807
Epoch 700 | Loss: 0.2368
Epoch 800 | Loss: 0.2125
Epoch 900 | Loss: 0.1860


In [12]:
modality_model.eval()
with torch.no_grad():
    preds = modality_model(combined_features[val_mask])
    predicted = preds.argmax(dim = 1)

    val_labels = modality_labels[val_mask].numpy()
    val_preds = predicted.numpy()
    print("=== Val Set ===")
    print(f"Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
    print(f"\n{classification_report(val_labels, val_preds, target_names=['visual', 'hands_on', 'analytical', 'exploratory'])}")

=== Val Set ===
Macro F1: 0.9315

              precision    recall  f1-score   support

      visual       0.92      0.95      0.93        75
    hands_on       0.93      0.97      0.95        73
  analytical       0.97      0.97      0.97        91
 exploratory       0.91      0.84      0.87        61

    accuracy                           0.94       300
   macro avg       0.93      0.93      0.93       300
weighted avg       0.94      0.94      0.94       300



In [13]:
modality_model.eval()
with torch.no_grad():
    preds = modality_model(combined_features[test_mask])
    predicted = preds.argmax(dim=1)
    
    test_labels = modality_labels[test_mask].numpy()
    test_preds = predicted.numpy()
    
    print("=== Test Set ===")
    print(f"Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"\n{classification_report(test_labels, test_preds, target_names=['visual', 'hands_on', 'analytical', 'exploratory'])}")

=== Test Set ===
Macro F1: 0.9331

              precision    recall  f1-score   support

      visual       0.92      0.96      0.94        89
    hands_on       0.95      0.95      0.95        81
  analytical       1.00      0.97      0.99        70
 exploratory       0.86      0.85      0.86        60

    accuracy                           0.94       300
   macro avg       0.93      0.93      0.93       300
weighted avg       0.94      0.94      0.94       300



In [14]:
torch.save(modality_model.state_dict(), 'models/task_c_model_new.pt')
print("Task C model saved!")

Task C model saved!
